# E005 sur GPU (Google Colab)

Entraîne `configs/e005_fulldata.yaml` (E004 avec les 80 pulses d'entraînement par configuration), l'évalue sur
toute la validation (20 pulses par configuration, 13 SNR) et copie le dossier du run sur Google Drive.

**Avant de lancer**, le dossier `MyDrive/radarml/` de ton Google Drive doit contenir :

- `radarml_code.zip` : le code du dépôt (branche `main`) ;
- `recorded_dataset.zip` : les données d'entraînement et de validation (2,5 Go).

Puis : `Exécution > Modifier le type d'exécution > GPU T4`, et `Exécution > Tout exécuter`.

**Règle** : seules les données d'entraînement et de validation vont sur Drive et Colab. Le blind test n'y va
jamais.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/radarml"      # contient radarml_code.zip et recorded_dataset.zip
CONFIG = "configs/e005_fulldata.yaml"

## 1. Vérifier le GPU

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relance"
print("PyTorch", torch.__version__, "| GPU :", torch.cuda.get_device_name(0))

## 2. Connecter Google Drive et installer le code

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
for name in ("radarml_code.zip", "recorded_dataset.zip"):
    assert os.path.isfile(f"{DRIVE_DIR}/{name}"), f"Fichier manquant sur Drive : {DRIVE_DIR}/{name}"

!rm -rf /content/repo && mkdir -p /content/repo
!unzip -q "{DRIVE_DIR}/radarml_code.zip" -d /content/repo
%cd /content/repo
!pip install --quiet -r requirements.txt && pip install --quiet -e .

## 3. Préparer les données (environ 5 min)

In [ ]:
# Copie sur le disque de Colab : lire 660 fichiers directement sur Drive ralentirait l'entraînement.
!cp "{DRIVE_DIR}/recorded_dataset.zip" /content/recorded_dataset.zip
!python scripts/setup_data.py --zip /content/recorded_dataset.zip --skip-file-hashes
!python -m pytest -q

## 4. Entraîner E005

La sortie affiche une ligne par époque (8 au total). Après la dernière, le script fait une évaluation rapide
puis affiche `SCORE VALIDATION ...` et `Run : ...`. Le dossier du run est copié sur Drive dès la fin.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

experiment = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))["experiment"]
subprocess.run([sys.executable, "scripts/train.py", "--config", CONFIG, "--device", "cuda"], check=True)
run = sorted(Path("runs").glob(f"{experiment}_*"))[-1]
drive_run = Path(DRIVE_DIR) / "runs" / run.name
shutil.copytree(run, drive_run, dirs_exist_ok=True)
print("Copié sur Drive :", drive_run)

## 5. Évaluer sur toute la validation

In [ ]:
subprocess.run([sys.executable, "scripts/evaluate.py", "--run", str(run), "--device", "cuda"], check=True)
shutil.copytree(run, drive_run, dirs_exist_ok=True)
print("Copié sur Drive :", drive_run)

## 6. Terminé

Télécharge le dossier `MyDrive/radarml/runs/E005_fulldata_...` et place-le dans le dossier `runs\` du dépôt sur
ton PC. Les temps d'inférence de ce run sont des temps GPU : ils ne se comparent pas aux runs CPU du journal.